# 02_TS
---

![](img/ts.png)

* Arma la Training Strategy, parte del dataset con feature engineering
1. dataset de future, donde se va a aplicar el modelo, son los datos que NO tiene clase
2. dataset de la train_strategy, donde estan marcados los registros de training, validation y testing. Ppuede no considerar algunos meses, hacer undersampling de las clases, etc. Hace falta mucha creatividad para esta estapa, decidir donde entrar es MUY ESTRATEGICO y esta sometido a las limitaciones de procesamiento
3. dataset de  train_final  donde se va a entrenar el modelo final una vez que se tengan los mejores hiperparametros de la Bayesian Optimization

In [10]:
# se limpia memoria
rm(list=ls())
gc()

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,741026,39.6,1447452,77.4,NA,1447452,77.4
Vcells,1383224,10.6,8388608,64.0,16384,2514552,19.2


In [11]:
library(yaml)
require("data.table")

In [12]:
setwd("/Users/ppando/Materias/cs-datos/especializacion/ciencia-datos/exp/")

In [13]:
PARAMS <- read_yaml("02_TS_prxs_01a.yml")

In [14]:
# particionar en el dataset una seccion del yaml

aplicar_particion  <- function(seccion) {
  columna_nueva  <- paste0("part_", seccion)
  dataset[  , (columna_nueva) := 0L]

  if (length(PARAMS$param[[seccion]]$periodos) > 0) {
    dataset[get(PARAMS$param$const$periodo) %in% PARAMS$param[[seccion]]$periodos , (columna_nueva)  := 1L]
  } else {
     dataset[get( PARAMS$param$const$periodo) >= PARAMS$param[[seccion]]$rango$desde  &  get(PARAMS$param$const$periodo) <= PARAMS$param[[seccion]]$rango$hasta,
              (columna_nueva)  := 1L]
  }

  if (length(PARAMS$param[[seccion]]$excluir) > 0) {
    dataset[get(PARAMS$param$const$periodo ) %in%  PARAMS$param[[seccion]]$excluir, (columna_nueva) := 0L]
  }

  if ("undersampling" %in% names(PARAMS$param[[seccion]])) {
    for (clase_valor in PARAMS$param[[seccion]]$undersampling) {
       dataset[get(columna_nueva)==1L & get(PARAMS$param$const$clase) == clase_valor$clase & part_azar > clase_valor$prob,
                (columna_nueva) := 0L ]                  
    }
  }

}

## Main

In [15]:
# se carga el dataset
nom_arch  <- PARAMS$param$files$input$dentrada
dataset   <- fread(nom_arch)

In [16]:
# se ordena dataset por <foto_mes, numero_de_cliente> 
setorderv(dataset, PARAMS$param$const$campos_sort)

set.seed(PARAMS$param$semilla)
dataset[ , part_azar  := runif(nrow(dataset))]   # se genera variable azar, con la que se va a particionar

In [17]:
# se hacen las particiones de cada seccion
# las secciones son future, train, validate, test, train_final

for (seccion in PARAMS$param$const$secciones) {
  aplicar_particion(seccion)
}

dataset[ , part_azar := NULL]  # ya no se necesita el campo  part_azar

psecciones  <- paste0("part_", PARAMS$param$const$secciones)

# generar el archivo de control, que DEBE ser revisado
tb_control  <- dataset[ , .N, psecciones]

fwrite( tb_control, file = PARAMS$param$files$output$control, sep = "\t" )

In [18]:
# grabar Future
if (0 < dataset[part_future>0, .N]) {
  fwrite(dataset[part_future>0,
                 setdiff(colnames(dataset), c(psecciones, PARAMS$param$const$clase)),
                 with = FALSE],
        file = PARAMS$param$files$output$future_data,
        logical01 = TRUE,
        sep = "," )
}

In [19]:
# grabar train_strategy
if (0 < dataset[part_train>0 | part_validate>0 | part_test>0, .N]) {
  fwrite(dataset[part_train>0 | part_validate>0 | part_test>0,
                   setdiff(colnames(dataset) , c("part_future", "part_train_final")),
                   with = FALSE],
          file = PARAMS$param$files$output$train_strategy,
          logical01 = TRUE,
          sep = "," )
}

In [20]:
# grabar train_final
if (0 < dataset[part_train_final > 0, .N]) {
  fwrite(dataset[part_train_final > 0,
                   setdiff(colnames(dataset), psecciones),
                   with = FALSE],
          file = PARAMS$param$files$output$train_final,
          logical01 = TRUE,
          sep = "," )
}